# Lenguajes y Autómatas II (SCD-1016)
## Sesión 04 · Anatomía de un Árbol Sintáctico Abstracto (AST)

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Cómo se guarda una expresión matemática para recorrerla sin perder su estructura?
>
> **Objetivo de la sesión:** Modelar expresiones como nodos enlazados (número u operación binaria) y evaluarlas recorriendo el árbol.

**Entregable:** tu AST en artesano_sesion04_c.c, el harness en verde y la bitácora de la sesión

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)

> ¿Dudas de C? Consulta la [guía de C](https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2/blob/main/docs/guia-c.md) antes de empezar.


## 1. ¿Qué vas a construir?

Construyes nodos numéricos y binarios, los enlazas y el harness recorre el árbol para comprobar su forma y su valor. Solo se evalúan +, - y *.

Al final de la sesión tendrás: **Un AST mínimo en C con el que puedes construir 3+5*2 y obtener 13 al evaluarlo.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 04](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion04.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion04.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion04.h"

ASTNode *crear_numero(int v);
ASTNode *crear_binario(char op, ASTNode *izq, ASTNode *der);
int evaluar(ASTNode *n);
```

- TipoNodo distingue número de operación; no uses campos mágicos.
- Cada nodo se reserva en el heap.
- evaluar(NULL) devuelve -1 y no debe caerse.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion04_c.c` junto con `artesano_sesion04_c.c`
> (o `ia_sesion04_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `evaluar(nodo 42)` | 42 |
| 2 | `evaluar(3+5*2)` | 13 |
| 3 | `hijo izquierdo de (* 5 2)` | 5 |
| 4 | `operador guardado` | `*` |
| 5 | `evaluar(NULL)` | -1 |
| 6 | `evaluar(10-4)` | 6 |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 04 - Nodo binario de AST ===
Caso                               esperado:              obtenido:              resultado
[ 1] evaluar(nodo 42)                   esperado: 42           obtenido: 42           OK
[ 2] evaluar(3+5*2)                     esperado: 13           obtenido: 13           OK
[ 3] hijo izquierdo de (* 5 2)          esperado: 5            obtenido: 5            OK
[ 4] operador guardado                  esperado: *            obtenido: *            OK
[ 5] evaluar(NULL)                      esperado: -1           obtenido: -1           OK
[ 6] evaluar(10-4)                      esperado: 6            obtenido: 6            OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion04.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] crear_binario respeta el orden izquierdo/derecho de los hijos.
- [ ] evaluar aplica el operador después de evaluar ambos hijos.
- [ ] evaluar(NULL) devuelve -1 sin provocar fallo de segmentación.


## Paso 0 · Entorno


In [1]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


✅ Entorno listo:
gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion04_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [9]:
%%writefile api_sesion04.h
/* api_sesion04.h - contrato de la Sesion 04
 * Nodo binario de AST
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION04_H
#define API_SESION04_H

typedef enum { NODO_NUM, NODO_BINOP } TipoNodo;
typedef struct ASTNode {
    TipoNodo tipo;
    int valor;
    char op;
    struct ASTNode *izq;
    struct ASTNode *der;
} ASTNode;

ASTNode *crear_numero(int v);
ASTNode *crear_binario(char op, ASTNode *izq, ASTNode *der);
int evaluar(ASTNode *n);

#endif /* API_SESION04_H */


Overwriting api_sesion04.h


In [10]:
%%writefile artesano_sesion04_c.c
/* artesano_sesion04_c.c - Fase 1: El Artesano (sin IA)
 * Nodo binario de AST.
 *
 * Implementa lo que declara api_sesion04.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */
#include "api_sesion04.h"
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

ASTNode *crear_numero(int v) {
    ASTNode *nodo = (ASTNode *)malloc(sizeof(ASTNode));
    if (nodo != NULL) {
        nodo->tipo = NODO_NUM;
        nodo->valor = v;
        nodo->op = '\0';
        nodo->izq = NULL;
        nodo->der = NULL;
    }
    return nodo;
}

ASTNode *crear_binario(char op, ASTNode *izq, ASTNode *der) {
    ASTNode *nodo = (ASTNode *)malloc(sizeof(ASTNode));
    if (nodo != NULL) {
        nodo->tipo = NODO_BINOP;
        nodo->valor = 0;
        nodo->op = op;
        nodo->izq = izq;
        nodo->der = der;
    }
    return nodo;
}

int evaluar(ASTNode *n) {
    // Caso base: si el nodo es NULL, retorna -1 según la auditoría
    if (n == NULL) {
        return -1;
    }

    // Caso base: si es un nodo numérico, retorna su valor
    if (n->tipo == NODO_NUM) {
        return n->valor;
    }

    // Caso recursivo: si es un operador binario, evalúa ambos hijos
    if (n->tipo == NODO_BINOP) {
        int val_izq = evaluar(n->izq);
        int val_der = evaluar(n->der);

        switch (n->op) {
            case '+': return val_izq + val_der;
            case '-': return val_izq - val_der;
            case '*': return val_izq * val_der;
            default:
                printf("Error: Operador desconocido '%c'\n", n->op);
                return 0;
        }
    }

    return -1;
}

Overwriting artesano_sesion04_c.c


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion04_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


C — Contexto: C11, gcc -Wall -Wextra -Werror, Linux. El archivo api_sesion04.h ya existe y no se puede modificar. Prohibido usar system().

R — Rol: ingeniero de compiladores con experiencia en C.

E — Especificación: Construye nodos numéricos y binarios, enlázalos correctamente y permite que el harness recorra el árbol para comprobar su forma y valor. Ten en cuenta que solo se evalúan +, - y *, y se debe contemplar la seguridad ante punteros nulos en evaluar(NULL) retornando -1.

O — Output: Únicamente el código fuente en C implementando las funciones declaradas en api_sesion04.h (crear_numero, crear_binario y evaluar). No incluyas la función main() ni explicaciones adicionales, asegurando que compile de forma limpia sin advertencias bajo -Wall -Wextra -Werror.

In [13]:
%%writefile ia_sesion04_c.c
/* ia_sesion04_c.c - Fase 2: El Cientifico
 *
 * Implementacion del AST para la Sesion 04 generada de forma precisa.
 */
#include "api_sesion04.h"
#include <stdlib.h>
#include <stdio.h>

ASTNode *crear_numero(int v) {
    ASTNode *nodo = (ASTNode *)malloc(sizeof(ASTNode));
    if (nodo != NULL) {
        nodo->tipo = NODO_NUM;
        nodo->valor = v;
        nodo->op = '\0';
        nodo->izq = NULL;
        nodo->der = NULL;
    }
    return nodo;
}

ASTNode *crear_binario(char op, ASTNode *izq, ASTNode *der) {
    ASTNode *nodo = (ASTNode *)malloc(sizeof(ASTNode));
    if (nodo != NULL) {
        nodo->tipo = NODO_BINOP;
        nodo->valor = 0;
        nodo->op = op;
        nodo->izq = izq;
        nodo->der = der;
    }
    return nodo;
}

int evaluar(ASTNode *n) {
    if (n == NULL) {
        return -1;
    }

    if (n->tipo == NODO_NUM) {
        return n->valor;
    }

    if (n->tipo == NODO_BINOP) {
        int val_izq = evaluar(n->izq);
        int val_der = evaluar(n->der);

        switch (n->op) {
            case '+':
                return val_izq + val_der;
            case '-':
                return val_izq - val_der;
            case '*':
                return val_izq * val_der;
            default:
                return 0;
        }
    }

    return -1;
}

Writing ia_sesion04_c.c


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [11]:
%%writefile harness_sesion04_c.c
/* harness_sesion04_c.c - auditoria automatica de la Sesion 04
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion04_c.c artesano_sesion04_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion04.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_str(const char *caso, const char *esperado,
                          const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12s obtenido: %-12s %s\n",
           probadas, caso, esperado, obtenido ? obtenido : "(NULL)",
           ok ? "OK" : "FALLA");
}

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 04 - Nodo binario de AST ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { ASTNode *n = crear_numero(42); verificar_int("evaluar(nodo 42)", (long)(42), (long)(evaluar(n))); }
    { ASTNode *n = crear_binario('+', crear_numero(3), crear_binario('*', crear_numero(5), crear_numero(2))); verificar_int("evaluar(3+5*2)", (long)(13), (long)(evaluar(n))); }
    { ASTNode *n = crear_binario('*', crear_numero(5), crear_numero(2)); verificar_int("hijo izquierdo de (* 5 2)", (long)(5), (long)((n && n->izq ? n->izq->valor : -1))); }
    { ASTNode *n = crear_binario('*', crear_numero(5), crear_numero(2)); char b[2]; b[0] = n ? n->op : '?'; b[1] = '\0'; verificar_str("operador guardado", "*", b); }
    {  verificar_int("evaluar(NULL)", (long)(-1), (long)(evaluar(NULL))); }
    { ASTNode *n = crear_binario('-', crear_numero(10), crear_numero(4)); verificar_int("evaluar(10-4)", (long)(6), (long)(evaluar(n))); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion04.md" "\n");
    return 0;
}


Overwriting harness_sesion04_c.c


In [12]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion04_c.c artesano_sesion04_c.c -o test_artesano && ./test_artesano

=== AUDITORIA SESION 04 - Nodo binario de AST ===
Caso                               esperado:              obtenido:              resultado
[ 1] evaluar(nodo 42)                   esperado: 42           obtenido: 42           OK
[ 2] evaluar(3+5*2)                     esperado: 13           obtenido: 13           OK
[ 3] hijo izquierdo de (* 5 2)          esperado: 5            obtenido: 5            OK
[ 4] operador guardado                  esperado: *            obtenido: *            OK
[ 5] evaluar(NULL)                      esperado: -1           obtenido: -1           OK
[ 6] evaluar(10-4)                      esperado: 6            obtenido: 6            OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion04.md


In [14]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion04_c.c ia_sesion04_c.c -o test_ia && ./test_ia

=== AUDITORIA SESION 04 - Nodo binario de AST ===
Caso                               esperado:              obtenido:              resultado
[ 1] evaluar(nodo 42)                   esperado: 42           obtenido: 42           OK
[ 2] evaluar(3+5*2)                     esperado: 13           obtenido: 13           OK
[ 3] hijo izquierdo de (* 5 2)          esperado: 5            obtenido: 5            OK
[ 4] operador guardado                  esperado: *            obtenido: *            OK
[ 5] evaluar(NULL)                      esperado: -1           obtenido: -1           OK
[ 6] evaluar(10-4)                      esperado: 6            obtenido: 6            OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion04.md


## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion04.md` a `auditorias/auditoria_sesion04.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
